[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module5/05-clustering.ipynb)

# Module 5 — Lesson 5: Clustering

**Module:** 5 — Machine Learning Foundations | **Time:** 25 minutes

## Learning Objectives

By the end of this lesson you will be able to:

- Apply K-Means clustering and choose k using the elbow method and silhouette score
- Use DBSCAN for density-based clustering and handle noise points
- Build hierarchical (agglomerative) clustering and interpret a dendrogram
- Compare clustering algorithms on blobs, moons, and circles datasets
- Know when to use each algorithm based on data shape

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons, make_circles, load_iris
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('Libraries loaded.')

## 1. K-Means Clustering

K-Means partitions n observations into k clusters by:
1. Randomly initialising k centroids
2. Assigning each point to the nearest centroid
3. Recomputing centroids as the mean of assigned points
4. Repeating steps 2–3 until convergence

The objective minimised is the **within-cluster sum of squares (inertia)**.

In [ ]:
# Generate blob data with 4 true clusters
X_blobs, y_true = make_blobs(n_samples=400, centers=4, cluster_std=0.9, random_state=42)
X_scaled = StandardScaler().fit_transform(X_blobs)

# Fit K-Means with k=4
km = KMeans(n_clusters=4, random_state=42, n_init='auto')
labels = km.fit_predict(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], c=y_true, cmap='tab10', s=20, alpha=0.7)
axes[0].set_title('True Labels (Blobs)')

axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels, cmap='tab10', s=20, alpha=0.7)
axes[1].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
                s=250, marker='X', c='red', zorder=5, edgecolors='black', label='Centroids')
axes[1].set_title(f'K-Means (k=4)  |  Inertia={km.inertia_:.1f}')
axes[1].legend()

for ax in axes:
    ax.set_xlabel('Feature 1 (scaled)')
    ax.set_ylabel('Feature 2 (scaled)')

plt.tight_layout()
plt.show()
print(f'Adjusted Rand Index (vs true labels): {adjusted_rand_score(y_true, labels):.4f}')

## 2. Elbow Method and Silhouette Score for Choosing k

- **Elbow method** — plot inertia vs k; look for the "elbow" where adding more clusters yields diminishing returns
- **Silhouette score** — measures how similar a point is to its own cluster vs other clusters; ranges from -1 (wrong cluster) to +1 (dense, well-separated)

In [ ]:
k_range = range(2, 11)
inertias   = []
silhouettes = []

for k in k_range:
    km_k = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels_k = km_k.fit_predict(X_scaled)
    inertias.append(km_k.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels_k))

best_k_sil = k_range[np.argmax(silhouettes)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(list(k_range), inertias, 'o-', color='steelblue', lw=2)
axes[0].set_xlabel('Number of clusters k')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Curve')
axes[0].axvline(4, linestyle='--', color='red', label='True k=4')
axes[0].legend()

axes[1].plot(list(k_range), silhouettes, 's-', color='tomato', lw=2)
axes[1].set_xlabel('Number of clusters k')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs k')
axes[1].axvline(best_k_sil, linestyle='--', color='green', label=f'Best k={best_k_sil}')
axes[1].legend()

plt.suptitle('Choosing k for K-Means', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Best k by silhouette: {best_k_sil}  |  Silhouette score: {max(silhouettes):.4f}')

## 3. K-Means on Real Data — Iris Dataset

Apply K-Means to the Iris dataset (4D) and visualise the results in PCA-reduced 2-D space.

In [ ]:
from sklearn.decomposition import PCA

iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)

km_iris = KMeans(n_clusters=3, random_state=42, n_init='auto')
labels_iris = km_iris.fit_predict(X_iris)

X_pca = PCA(n_components=2).fit_transform(X_iris)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, labels, title in zip(axes,
                              [iris.target, labels_iris],
                              ['True Species', 'K-Means (k=3)']):
    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', s=30, alpha=0.8)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(title)

plt.suptitle('K-Means on Iris Dataset (PCA-reduced)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Adjusted Rand Index: {adjusted_rand_score(iris.target, labels_iris):.4f}')
print(f'Silhouette Score:    {silhouette_score(X_iris, labels_iris):.4f}')

## 4. DBSCAN — Density-Based Clustering

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) does not require specifying k. Instead it requires:

- `eps` — the radius of the neighbourhood
- `min_samples` — the minimum number of points to form a core point

Points that are not reachable from any core point are labelled as **noise** (label = -1). DBSCAN excels at discovering arbitrary-shaped clusters.

In [ ]:
# Make non-convex shapes where K-Means fails
X_moons, _ = make_moons(n_samples=300, noise=0.08, random_state=42)
X_moons_sc = StandardScaler().fit_transform(X_moons)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# K-Means (will fail)
km_m = KMeans(n_clusters=2, random_state=42, n_init='auto')
axes[0].scatter(X_moons_sc[:, 0], X_moons_sc[:, 1], c=km_m.fit_predict(X_moons_sc), cmap='tab10', s=20)
axes[0].set_title('K-Means (fails on moons)')

# DBSCAN with good params
db = DBSCAN(eps=0.3, min_samples=5)
labels_db = db.fit_predict(X_moons_sc)
n_noise = (labels_db == -1).sum()
axes[1].scatter(X_moons_sc[:, 0], X_moons_sc[:, 1], c=labels_db, cmap='tab10', s=20)
axes[1].set_title(f'DBSCAN eps=0.3 (noise={n_noise})')

# DBSCAN with bad eps (too large)
db2 = DBSCAN(eps=1.5, min_samples=5)
labels_db2 = db2.fit_predict(X_moons_sc)
axes[2].scatter(X_moons_sc[:, 0], X_moons_sc[:, 1], c=labels_db2, cmap='tab10', s=20)
axes[2].set_title(f'DBSCAN eps=1.5 (merges clusters)')

for ax in axes:
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('DBSCAN vs K-Means on Moon-Shaped Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'DBSCAN clusters found: {len(set(labels_db)) - (1 if -1 in labels_db else 0)}')
print(f'Noise points: {n_noise}')

## 5. Hierarchical (Agglomerative) Clustering and Dendrogram

Agglomerative clustering starts with each point as its own cluster and repeatedly merges the two closest clusters. The **dendrogram** visualises the entire merge history — cut it at the desired height to get any number of clusters.

In [ ]:
# Use a subset for readability
X_sub = X_scaled[:80]

# scipy linkage for dendrogram
Z = linkage(X_sub, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dendrogram
dendrogram(Z, ax=axes[0], truncate_mode='lastp', p=20,
           leaf_rotation=90, leaf_font_size=8, show_contracted=True)
axes[0].set_title('Dendrogram (Ward linkage, truncated)')
axes[0].set_xlabel('Sample index')
axes[0].set_ylabel('Distance')
axes[0].axhline(y=4, color='red', linestyle='--', label='Cut → 4 clusters')
axes[0].legend()

# Agglomerative Clustering
agg = AgglomerativeClustering(n_clusters=4, linkage='ward')
labels_agg = agg.fit_predict(X_scaled)
axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels_agg, cmap='tab10', s=20, alpha=0.7)
axes[1].set_title('Agglomerative Clustering (k=4, Ward)')
axes[1].set_xlabel('Feature 1 (scaled)')
axes[1].set_ylabel('Feature 2 (scaled)')

plt.tight_layout()
plt.show()
print(f'Adjusted Rand Index (vs true): {adjusted_rand_score(y_true, labels_agg):.4f}')
print(f'Silhouette Score:              {silhouette_score(X_scaled, labels_agg):.4f}')

## 6. Algorithm Comparison — Blobs, Moons, Circles

Different algorithms are suited to different data shapes. This benchmark helps build intuition.

In [ ]:
datasets = {
    'Blobs':   make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42),
    'Moons':   make_moons(n_samples=300, noise=0.07, random_state=42),
    'Circles': make_circles(n_samples=300, factor=0.5, noise=0.06, random_state=42)
}

algorithms = {
    'K-Means (k=3)':     KMeans(n_clusters=3, random_state=42, n_init='auto'),
    'DBSCAN':            DBSCAN(eps=0.35, min_samples=8),
    'Agglomerative (k=3)': AgglomerativeClustering(n_clusters=3)
}

fig, axes = plt.subplots(len(datasets), len(algorithms), figsize=(14, 10))

for row, (ds_name, (X_ds, y_ds)) in enumerate(datasets.items()):
    X_ds_sc = StandardScaler().fit_transform(X_ds)
    for col, (alg_name, alg) in enumerate(algorithms.items()):
        ax = axes[row][col]
        labels_alg = alg.fit_predict(X_ds_sc)
        n_clusters = len(set(labels_alg)) - (1 if -1 in labels_alg else 0)
        noise_n = (labels_alg == -1).sum()
        ax.scatter(X_ds_sc[:, 0], X_ds_sc[:, 1], c=labels_alg, cmap='tab10', s=12, alpha=0.7)
        ax.set_title(f'{ds_name}\n{alg_name}\nclusters={n_clusters}, noise={noise_n}', fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle('Clustering Algorithm Comparison Across Dataset Shapes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('\nKey insight: DBSCAN handles non-convex shapes; K-Means assumes spherical clusters.')

## Practice Exercises

**Exercise 1 — Customer Segmentation Simulation**
Generate a dataset with 500 samples and 5 features using `make_blobs(centers=5)`. Scale the data. Use K-Means with the elbow method to find the optimal k. Report the silhouette score for the chosen k and describe what each cluster's centroid (before scaling) represents.

**Exercise 2 — DBSCAN Parameter Sensitivity**
Using the `make_circles` dataset, run DBSCAN over a grid of `eps` values `[0.1, 0.2, 0.3, 0.5, 0.8]` and `min_samples` values `[3, 5, 10]`. Create a heatmap showing the number of clusters found for each combination. Identify the configuration that recovers exactly 2 clusters with zero noise.

**Exercise 3 — Hierarchical Linkage Comparison**
Using the blobs dataset from section 1 (X_scaled), fit `AgglomerativeClustering` with `linkage='ward'`, `'complete'`, `'average'`, and `'single'` — each with k=4. Compare their adjusted rand index against true labels. Which linkage method performs best?